# Schema Evolution & Breaking Changes

An upstream producer renames columns, changes types, or splits fields between releases.
Without a schema contract, your downstream pipeline discovers the breakage at runtime —
often as a silent wrong result rather than a loud error.

This notebook demonstrates the problem with two Parquet schema versions, builds a
breaking-change detector with PyArrow, and implements a contract-based validator that
catches issues before downstream logic runs.

In [ ]:
!pip install pandas pyarrow duckdb --quiet

## Generate Producer V1 and V2 Data

The producer evolves their schema between two releases:

| V1 Column | V2 Column | Change |
| :--- | :--- | :--- |
| `full_name` (string) | `first_name` + `last_name` (string) | Split into two columns |
| `signup_date` (timestamp) | `signup_ts` (timestamp) | Renamed |
| `country` (string) | `country_code` (string) | Renamed + shortened values |
| `revenue` (float64) | `revenue_cents` (int64) | Changed type + semantics (dollars → cents) |

In [ ]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os
import tempfile

np.random.seed(42)

tmpdir = tempfile.mkdtemp()
v1_path = os.path.join(tmpdir, "users_v1.parquet")
v2_path = os.path.join(tmpdir, "users_v2.parquet")

# --- V1 schema: 5 000 rows ---
countries = ["United States", "Canada", "United Kingdom", "Germany", "France"]
first_names = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace", "Hank"]
last_names = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller"]

v1_df = pd.DataFrame({
    "user_id": range(1, 5001),
    "full_name": [f"{np.random.choice(first_names)} {np.random.choice(last_names)}" for _ in range(5000)],
    "signup_date": pd.date_range("2023-01-01", periods=5000, freq="h"),
    "country": np.random.choice(countries, size=5000),
    "revenue": np.round(np.random.uniform(0, 500, size=5000), 2),
})

v1_df.to_parquet(v1_path, index=False)

# --- V2 schema: 2 000 rows (new batch) ---
country_codes = {"United States": "US", "Canada": "CA", "United Kingdom": "GB",
                 "Germany": "DE", "France": "FR"}

v2_df = pd.DataFrame({
    "user_id": range(5001, 7001),
    "first_name": np.random.choice(first_names, size=2000),
    "last_name": np.random.choice(last_names, size=2000),
    "signup_ts": pd.date_range("2024-01-01", periods=2000, freq="h"),
    "country_code": np.random.choice(list(country_codes.values()), size=2000),
    "revenue_cents": np.random.randint(0, 50000, size=2000),  # cents, not dollars!
})

v2_df.to_parquet(v2_path, index=False)

print("V1 schema:")
print(pq.read_schema(v1_path))
print("\nV2 schema:")
print(pq.read_schema(v2_path))

## Anti-Pattern: Read Without Schema Validation

A downstream report pipeline that hardcodes column names from V1. When the producer
ships V2, this code either crashes (KeyError) or silently produces wrong results
(if a column with a similar name but different semantics exists).

In [ ]:
def process_revenue_report(parquet_path):
    """Downstream pipeline that assumes V1 schema."""
    df = pd.read_parquet(parquet_path)
    total_revenue = df["revenue"].sum()  # expects float dollars
    top_customers = df.nlargest(5, "revenue")[["full_name", "revenue"]]
    return total_revenue, top_customers

# Works fine on V1
total, top = process_revenue_report(v1_path)
print(f"V1 report — Total revenue: ${total:,.2f}")
display(top)

# Crashes on V2
try:
    process_revenue_report(v2_path)
except KeyError as e:
    print(f"\nV2 report — KeyError: {e}")
    print("The column 'revenue' doesn't exist in V2 — it's now 'revenue_cents'.")
    print("If the column had the same name but different units, this would be a SILENT bug.")

## Detecting Breaking Changes with PyArrow Schema Comparison

Before running any business logic, compare the incoming schema against the expected
schema. Report dropped columns, type changes, and new columns.

In [ ]:
def detect_breaking_changes(old_schema, new_schema):
    """Compare two PyArrow schemas and report breaking changes."""
    old_fields = {f.name: f.type for f in old_schema}
    new_fields = {f.name: f.type for f in new_schema}

    report = {"dropped": [], "type_changed": [], "added": []}

    for name, dtype in old_fields.items():
        if name not in new_fields:
            report["dropped"].append({"column": name, "old_type": str(dtype)})
        elif new_fields[name] != dtype:
            report["type_changed"].append({
                "column": name,
                "old_type": str(dtype),
                "new_type": str(new_fields[name]),
            })

    for name, dtype in new_fields.items():
        if name not in old_fields:
            report["added"].append({"column": name, "new_type": str(dtype)})

    return report


schema_v1 = pq.read_schema(v1_path)
schema_v2 = pq.read_schema(v2_path)

report = detect_breaking_changes(schema_v1, schema_v2)

print("=== Breaking Change Report ===")
print(f"\nDropped columns ({len(report['dropped'])}) — BREAKING:")
for d in report["dropped"]:
    print(f"  - {d['column']} ({d['old_type']})")

print(f"\nType changes ({len(report['type_changed'])}) — POTENTIALLY BREAKING:")
for t in report["type_changed"]:
    print(f"  - {t['column']}: {t['old_type']} -> {t['new_type']}")

print(f"\nNew columns ({len(report['added'])}) — non-breaking:")
for a in report["added"]:
    print(f"  + {a['column']} ({a['new_type']})")

## Correct Approach: Define a Contract and Validate on Read

A **data contract** is a formal definition of the expected schema. Validate incoming
data against the contract **before** any downstream logic runs. If the contract is
violated, fail loudly and immediately.

In [ ]:
# Define the contract as a PyArrow schema
revenue_contract = pa.schema([
    pa.field("user_id", pa.int64(), nullable=False),
    pa.field("full_name", pa.string(), nullable=False),
    pa.field("signup_date", pa.timestamp("ns"), nullable=False),
    pa.field("country", pa.string(), nullable=True),
    pa.field("revenue", pa.float64(), nullable=False),
])


def validate_against_contract(parquet_path, contract):
    """Validate a Parquet file against a schema contract."""
    actual = pq.read_schema(parquet_path)
    actual_fields = {f.name: f for f in actual}
    violations = []

    for expected_field in contract:
        if expected_field.name not in actual_fields:
            violations.append(f"MISSING: required column '{expected_field.name}' not found")
        else:
            actual_field = actual_fields[expected_field.name]
            if actual_field.type != expected_field.type:
                violations.append(
                    f"TYPE MISMATCH: '{expected_field.name}' expected {expected_field.type}, "
                    f"got {actual_field.type}"
                )

    if violations:
        print(f"CONTRACT VIOLATION — {len(violations)} issue(s):")
        for v in violations:
            print(f"  - {v}")
        return False
    else:
        print("Contract satisfied.")
        return True


print("=== Validate V1 against contract ===")
validate_against_contract(v1_path, revenue_contract)

print("\n=== Validate V2 against contract ===")
validate_against_contract(v2_path, revenue_contract)

print("\nV2 fails validation BEFORE any business logic runs — exactly what we want.")

In [ ]:
# DuckDB can also enforce schema expectations on read
import duckdb

con = duckdb.connect()

# Define expected columns in a view
try:
    con.execute(f"""
        CREATE VIEW revenue_report AS
        SELECT user_id, full_name, signup_date, country, revenue
        FROM read_parquet('{v2_path}')
    """)
    con.execute("SELECT * FROM revenue_report LIMIT 1").fetchdf()
except duckdb.BinderException as e:
    print(f"DuckDB caught it: {e}")
    print("\nThe query references 'full_name' which doesn't exist in V2.")
    print("DuckDB's SQL binder acts as a natural schema contract.")

## Takeaways

| Strategy | Catches | Tooling in Production |
| :--- | :--- | :--- |
| PyArrow schema comparison | Dropped/renamed columns, type changes | CI/CD pipeline check before deploy |
| Contract validation on read | Missing required columns, wrong types, nullability | Great Expectations, Soda, dbt contracts |
| SQL-based schema enforcement (DuckDB, Spark) | Column existence at query compile time | Spark `schema()` on read, DuckDB views |
| Schema Registry (Avro/Protobuf) | Backward/forward compatibility at serialization time | Confluent Schema Registry, AWS Glue |

**Key principle:** treat schema changes like API changes. A column rename or type change
is a **breaking change** that needs versioning, migration, and validation — not a silent
surprise at 3 AM.